In [1]:
# DPO algo 


# PG Loss = -R * log (policy (output | prompt ))

# We start with pairs here that is : correct answer, incorrect answer
# simply increase the prob of getting the correct answer and decrease the prob of getting bad answer 

# loss = [new(correct | prompt) - ref(correct | prompt)] - [new(incorrect | prompt) - ref(incorrect | prompt)]

# scaled_loss =  Beta * loss
# sigmoid : squashes no. into (0,1)
# BCE : -log sigmoid(scaled_loss)

# ---x---
# maintain : policy, ref_model

import torch 
from base import model 
from base import tokenizer
import torch.nn.functional as F
import copy 
import sys
import re


policy = model
reference = copy.deepcopy(model)

reference.eval()
for p in reference.parameters():
    p.requires_grad_(False)

PROMPT = "The capital of France is"
CORRECT_ANSWER = " Paris." # adding space for toy implementation
INCORRECT_ANSWER = " Paris"
 
G, STEPS, MAX_NEW, EPS, BETA, LR,K = 1, 100, 1, 0.2, 0.05, 1e-3, 2
opt = torch.optim.AdamW(policy.parameters(), lr=LR)


def token_logprobs(model, ids, prompt_len):
    logits = model(ids).logits[:, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, ids[:, 1:, None]).squeeze(-1)
    return lp[:, prompt_len-1:]



Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [2]:

device = next(policy.parameters()).device
for step in range(STEPS):
    prompt_ids = tokenizer(PROMPT, return_tensors="pt").input_ids

    prompt_correct_ans_ids = tokenizer(PROMPT + CORRECT_ANSWER, return_tensors='pt').input_ids.repeat(G, 1)
    prompt_correct_ans_ids = prompt_correct_ans_ids.to(device)
    
    prompt_incorrect_ans_ids = tokenizer(PROMPT + INCORRECT_ANSWER, return_tensors='pt').input_ids.repeat(G, 1)
    prompt_incorrect_ans_ids = prompt_incorrect_ans_ids.to(device)
    
    # Policy-Loss
    with torch.no_grad():
        prompt_length = prompt_ids.shape[-1]
        ref_corr_lp = token_logprobs(reference, prompt_correct_ans_ids, prompt_length).sum(-1)
        ref_incorr_lp = token_logprobs(reference, prompt_incorrect_ans_ids, prompt_length).sum(-1)
        
    new_corr_lp = token_logprobs(policy, prompt_correct_ans_ids, prompt_length).sum(-1)
    new_incorr_lp = token_logprobs(policy, prompt_incorrect_ans_ids, prompt_length).sum(-1)
    
    margin = BETA * ((new_corr_lp - ref_corr_lp) - (new_incorr_lp - ref_incorr_lp))
    loss = -F.logsigmoid(margin)
    loss = loss.mean()
        
    opt.zero_grad()
    loss.backward()

    # gradient norm
    total_norm = 0.0
    for p in policy.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2

    total_norm = total_norm**0.5
    print(f"Total Gradient Norm: {total_norm}")
    
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    opt.step()

    if step % 10 == 0:
        print(f"step {step:3d}  loss : {loss.detach().item()}")

Total Gradient Norm: 1.0148113349324561
step   0  loss : 0.6932682991027832
Total Gradient Norm: 0.12369403707217211
Total Gradient Norm: 0.06873192423373548
Total Gradient Norm: 0.034610695312107954
Total Gradient Norm: 7.171948210241239e-07
Total Gradient Norm: 1.1165026208288743e-06
Total Gradient Norm: 5.397235126208055e-08
Total Gradient Norm: 1.5263089117822106e-05
Total Gradient Norm: 1.8909239263710312e-08
Total Gradient Norm: 0.0003471374602777899
Total Gradient Norm: 1.2550788627400176e-06
step  10  loss : 0.6778541803359985
Total Gradient Norm: 1.6770745064858059e-06
Total Gradient Norm: 0.00018597605164192497
Total Gradient Norm: 0.00030713213993404265
Total Gradient Norm: 3.1162261315778184e-05
Total Gradient Norm: 0.0019476475332893645
Total Gradient Norm: 5.4068634128801096e-05
Total Gradient Norm: 6.499375654845833e-05
Total Gradient Norm: 0.013072419252303919
Total Gradient Norm: 2.210618215897002
Total Gradient Norm: 1.4563726404539847
step  20  loss : 0.6954080462455